# V9 uncertainty lab — understand MC Dropout in Visual Studio Code

**Your real V9 model. No NeuroScope. No training.** Run the notebook from the **root of your original V9 project**. 

We'll use three *real V9 test images* to understand why uncertainty sometimes catches mistakes and sometimes misses them. This is a teaching example; it is **not a population-level validation study**.

## Step 1 — Load the real EfficientNet-B4 model

The checkpoint stays unchanged. We use the same grayscale preprocessing as Step 09, and activate **only the classifier dropout** for the Monte Carlo passes.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from IPython.display import display
from src.step25_uncertainty_visual_demo import (
    CASES, VALIDATION_ENTROPY_CUTOFF, get_device,
    load_v9_model, predict_twenty_times, save_case_figure, save_comparison_figure
)
from src import config as C

print('Model checkpoint:', C.CHECKPOINT)
print('Checkpoint exists:', C.CHECKPOINT.is_file())
device = get_device('auto')
model = load_v9_model(C.CHECKPOINT, device)
print('Device:', device)
print('Validation threshold:', VALIDATION_ENTROPY_CUTOFF)

## Step 2 — Pick one MRI

Try `correct`, `uncertain`, or `confidently_wrong`. We will run the SAME image 20 times with dropout on, plot four class probabilities across 20 passes, and compute *predictive entropy*.

**Entropy is a measure of the spread of the predicted class probabilities, not the probability that the prediction is wrong.**

In [ ]:
case_name = 'uncertain'  # CHANGE: 'correct', 'uncertain', 'confidently_wrong'
info = CASES[case_name]
image_path = C.ROOT / 'uncertainty_lab_samples' / info['file']
result = predict_twenty_times(model, image_path, device, passes=20, seed=42)

print('True test label:', info['true_class'])
print('Deterministic predicted class:', result['det_prediction'])
print('Predicted correctly?', result['det_prediction'] == info['true_class'])
print('Deterministic confidence:', f"{result['det_confidence']:.2%}")
print('Predictive entropy:', f"{result['predictive_entropy']:.6f}")
print('Mutual information:', f"{result['mutual_information']:.6f}")
print('Refer for review?', result['referral'])

out = C.ROOT / 'results' / '25_uncertainty_visual_demo'
out.mkdir(parents=True, exist_ok=True)
chart = save_case_figure(case_name, info, result, out)
display(Image.open(chart))

## Step 3 — See the 20 predictions as actual numbers

Each row is a stochastic forward pass. The values are probabilities assigned to the four classes; they sum to 1. The prediction can remain the *same label* while the **probability distribution** changes.

In [ ]:
for i, probabilities in enumerate(result['mc_probs'], 1):
    print(f"Pass {i:02d}:", ', '.join(f'{name}={probabilities[j]:.4f}' for j, name in enumerate(C.CLASSES)))

print('\nMean probabilities:', dict(zip(C.CLASSES, result['mc_mean'].round(5))))

## Step 4 — Compare all three real test examples

1. **Correct + low uncertainty:** the model was consistent *and* correct.
2. **Wrong + high uncertainty:** uncertainty successfully flags a mistake.
3. **Wrong + low uncertainty:** uncertainty fails to flag a confident mistake.

**This last case is why uncertainty is a warning signal, never proof of truth.**

In [ ]:
records = {}
for case_name, info in CASES.items():
    image_path = C.ROOT / 'uncertainty_lab_samples' / info['file']
    r = predict_twenty_times(model, image_path, device, passes=20, seed=42)
    records[case_name] = {'info': info, 'result': r}
    print(f"{case_name:18s} | true={info['true_class']:10s} | pred={r['det_prediction']:10s} | "
          f"entropy={r['predictive_entropy']:.6f} | "
          f"{'REFER' if r['referral'] else 'NOT REFERRED'}")

chart = save_comparison_figure(records, out)
display(Image.open(chart))

## What we learned

**MC predictive entropy** and **mutual information** are not the same as calibrated confidence. Predictive entropy describes the ambiguity of the average class distribution, while mutual information is a proxy for variability due to stochastic model parameters.

The **0.00148676** threshold was selected on the original V9 **validation** split for the Step 24 90%-coverage experiment. In that *retrospective* full test study, it flagged **25 of 29** incorrect deterministic predictions, and missed **4**. The illustrative three-MRI experiment above is separate from that full-cohort performance evaluation.

The system cannot prove a prediction is correct. Even a low-uncertainty MRI may be confidently misclassified.